# 03 — Model Training

**Purpose:** Train and evaluate the group-stage and knockout models.
Chronological cross-validation, probability calibration, and feature importance analysis.
All model logic lives in `src/models.py` — this notebook drives it and records results.

**Inputs**
- `outputs/training_rows.parquet`
- `outputs/team_features_freeze.parquet`

**Outputs**
- `outputs/group_stage_model.joblib`
- `outputs/knockout_model.joblib`
- `outputs/penalty_model.joblib`
- `outputs/feature_importance_group.csv`
- `outputs/feature_importance_knockout.csv`
- `outputs/cv_results.csv`
- `outputs/charts/calibration_curve.png`

In [1]:
import sys
from pathlib import Path

PROJECT_ROOT = Path("__file__").resolve().parent.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import warnings
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from src.leakage_guard import check_no_synthetic_data, check_training_rows_chronological
from src.models import (
    GroupStageModel, KnockoutModel, PenaltyModel,
    ChronologicalCV, compute_rps, compute_brier, compute_directional_accuracy,
    fit_all_models,
    WIN, DRAW, LOSS, CALIBRATION_YEAR,
    PRETOURNAMENT_FEATURES,
)

OUTPUTS = PROJECT_ROOT / "outputs"
CHARTS  = OUTPUTS / "charts"
CHARTS.mkdir(parents=True, exist_ok=True)

print("Imports OK")

Imports OK


## Cell 2 — Load training data

In [2]:
training_rows  = pd.read_parquet(OUTPUTS / "training_rows.parquet")
team_features  = pd.read_parquet(OUTPUTS / "team_features_freeze.parquet")

print(f"Training rows : {training_rows.shape}")
print(f"Team features : {team_features.shape}")

# Re-run guards on loaded data
check_no_synthetic_data(training_rows)
check_training_rows_chronological(training_rows)
print("✓ Leakage guards re-confirmed on loaded data")

if "outcome" in training_rows.columns:
    dist = training_rows["outcome"].value_counts().sort_index()
    print(f"\nOutcome distribution: {dist.to_dict()}")

Training rows : (518, 37)
Team features : (48, 46)
✓ Leakage guards re-confirmed on loaded data

Outcome distribution: {0: 199, 1: 120, 2: 199}


## Cell 3 — Chronological CV fold structure

In [3]:
cv = ChronologicalCV()

if "wc_year" in training_rows.columns:
    fold_sizes = []
    for fold_year, train_years in cv.fold_structure:
        train_mask = training_rows["wc_year"].isin(train_years)
        test_mask  = training_rows["wc_year"] == fold_year
        fold_sizes.append({
            "held_out_year": fold_year,
            "train_rows":    int(train_mask.sum()),
            "test_rows":     int(test_mask.sum()),
        })
    print("CV fold sizes (Fold 7 / 2022 is calibration-only — not in this table):")
    pd.DataFrame(fold_sizes)

## Cell 4 — Train group-stage model with chronological CV

In [4]:
group_model = GroupStageModel()

print("Fitting GroupStageModel (chronological CV + calibration)...")
with warnings.catch_warnings():
    warnings.simplefilter("ignore")
    cv_results = group_model.fit(training_rows)

print(f"\nGroupStageModel fitted")

if cv_results is not None and hasattr(cv_results, '__iter__'):
    cv_df = pd.DataFrame(cv_results) if not isinstance(cv_results, pd.DataFrame) else cv_results
    print("\nCV results by fold:")
    display(cv_df)

    # Design spec: target accuracy > 69.5%; Elo baseline ≈ 66.7%
    if "accuracy" in cv_df.columns:
        mean_acc = cv_df["accuracy"].mean()
        print(f"\nMean CV accuracy: {mean_acc:.3f}  (target > 0.695, Elo baseline ≈ 0.667)")

Fitting GroupStageModel (chronological CV + calibration)...

GroupStageModel fitted


## Cell 5 — Calibration analysis on 2022 holdout

In [5]:
if "wc_year" in training_rows.columns:
    calibration_set = training_rows[training_rows["wc_year"] == CALIBRATION_YEAR].copy()
    print(f"Calibration holdout (2022): {len(calibration_set)} rows")

    feature_cols = [c for c in PRETOURNAMENT_FEATURES if c in calibration_set.columns]
    if feature_cols and "outcome" in calibration_set.columns:
        X_cal = calibration_set[feature_cols]
        y_cal = calibration_set["outcome"].values

        try:
            proba_cal = group_model.predict_proba(X_cal)

            fig, ax = plt.subplots(figsize=(7, 5))
            bins = np.linspace(0, 1, 6)
            bin_centres = (bins[:-1] + bins[1:]) / 2

            # WIN class calibration
            p_win = proba_cal[:, 2]  # WIN is class index 2
            actual_win = (y_cal == WIN).astype(float)
            bin_pred, bin_actual = [], []
            for lo, hi in zip(bins[:-1], bins[1:]):
                mask = (p_win >= lo) & (p_win < hi)
                if mask.sum() > 0:
                    bin_pred.append(p_win[mask].mean())
                    bin_actual.append(actual_win[mask].mean())

            ax.plot(bin_pred, bin_actual, "o-", label="Calibrated model (WIN class)")
            ax.plot([0, 1], [0, 1], "--k", alpha=0.4, label="Perfect calibration")
            ax.set_xlabel("Predicted probability")
            ax.set_ylabel("Actual win rate")
            ax.set_title(f"Calibration curve — 2022 holdout ({len(calibration_set)} rows)")
            ax.legend()
            plt.tight_layout()
            out_path = CHARTS / "calibration_curve.png"
            plt.savefig(out_path, dpi=150)
            plt.show()
            print(f"Saved → {out_path}")
        except Exception as e:
            print(f"Calibration plot skipped: {e}")

## Cell 6 — Train knockout model

In [10]:
knockout_model = KnockoutModel()

print("Fitting KnockoutModel...")
with warnings.catch_warnings():
    warnings.simplefilter("ignore")
    ko_results = knockout_model.fit(training_rows)

print(f"\nKnockoutModel fitted")

# Design spec: Stage Group B (QF/SF/Final) prefers LR over XGBoost
# due to the small (~49 match) training set for later-round matches
if hasattr(knockout_model, "stage_b_primary_model"):
    print(f"Stage B primary model: {type(knockout_model.stage_b_primary_model).__name__}")

Fitting KnockoutModel...

KnockoutModel fitted


## Cell 7 — Train penalty model

In [6]:
penalty_model = PenaltyModel()
# PenaltyModel is formula-based (Bayesian shrinkage + Elo + WC rate) — no fitting required.
print("PenaltyModel initialised (no training required — formula-based ensemble)")

PenaltyModel initialised (no training required — formula-based ensemble)


## Cell 8 — Feature importance

In [11]:
for model, name in [(group_model, "group"), (knockout_model, "knockout")]:
    try:
        importance = getattr(model, "feature_importance_", None)
        if importance is not None:
            out_path = OUTPUTS / f"feature_importance_{name}.csv"
            importance.to_csv(out_path)
            print(f"Saved → {out_path}")
            print(f"\nTop 10 {name} features:")
            display(importance.head(10))
        else:
            print(f"Feature importance for {name}: not available (XGBoost may not have been used)")
    except Exception as e:
        print(f"Feature importance for {name} skipped: {e}")

Saved → /Users/pranaavdhaksheshganesh/Downloads/FIFA_WC_2026_Project/outputs/feature_importance_group.csv

Top 10 group features:


,feature,gain
1,elo_win_expectancy,3.096498
2,elo_rating_delta,2.921433
0,elo_rating,2.181501
8,wc_win_rate_modern,2.139808
13,form_win_rate_last10,2.136644
10,wc_gd_per_game_modern,2.033458
9,wc_win_rate_knockout_modern,2.028064
11,wc_group_vs_knockout_uplift,1.806498
14,form_gd_last10,1.796164
3,elo_is_host,0.000000


Feature importance for knockout: not available (XGBoost may not have been used)


## Cell 9 — Sanity checks on known 2022 matches

In [12]:
# Design spec: model should assign >50% win prob to the higher-Elo team in ≥8/10 2022 matches
if "wc_year" in training_rows.columns:
    matches_2022 = training_rows[
        (training_rows["wc_year"] == 2022) &
        (training_rows.get("perspective", pd.Series("home", index=training_rows.index)) == "home")
    ].head(10)

    feature_cols = [c for c in PRETOURNAMENT_FEATURES if c in matches_2022.columns]
    if feature_cols and len(matches_2022) >= 5:
        try:
            proba = group_model.predict_proba(matches_2022[feature_cols])
            p_home_wins = proba[:, 2]  # P(WIN) for home perspective

            directional_ok = 0
            for i, (_, row) in enumerate(matches_2022.iterrows()):
                elo_delta = row.get("elo_rating_delta", 0)
                p_win     = p_home_wins[i]
                # Higher Elo team should be favoured
                if (elo_delta > 0 and p_win > 0.5) or (elo_delta < 0 and p_win < 0.5):
                    directional_ok += 1

            print(f"Directional accuracy on 2022 sample: {directional_ok}/{len(matches_2022)}")
            print("  (design spec target: ≥ 8/10)")
        except Exception as e:
            print(f"Sanity check skipped: {e}")

## Cell 10 — Save models

In [13]:
models_to_save = {
    "group_stage_model.joblib":  group_model,
    "knockout_model.joblib":     knockout_model,
    "penalty_model.joblib":      penalty_model,
}

for filename, model in models_to_save.items():
    path = OUTPUTS / filename
    joblib.dump(model, path)
    print(f"Saved → {path}")

print("\n✓ All models serialised")

Saved → /Users/pranaavdhaksheshganesh/Downloads/FIFA_WC_2026_Project/outputs/group_stage_model.joblib
Saved → /Users/pranaavdhaksheshganesh/Downloads/FIFA_WC_2026_Project/outputs/knockout_model.joblib
Saved → /Users/pranaavdhaksheshganesh/Downloads/FIFA_WC_2026_Project/outputs/penalty_model.joblib

✓ All models serialised
